# Amenability Analysis Deep Dive

This notebook provides a deep dive into amenability scoring, design heuristics, and failure mode classification for guiding architecture selection.

**Goal:** Understand how to evaluate whether a neural network architecture will work well on analog hardware.

**Amenability is a bidirectional metric:** it tells hardware architects which patterns their hardware should support, and neural architects which patterns to use for analog deployment.

## What is Amenability Score?

Amenability score is an empirical metric (0-1) that combines multiple factors to predict how well an architecture will work on analog hardware:

- **Analog FLOP fraction:** More analog compute = better (rewards physics-based computation)
- **Noise tolerance:** Higher sigma_10pct from sweeps = better (tolerates mismatch)
- **D/A boundary count:** Fewer boundaries = better (less energy + quantization error)
- **Dynamics penalty:** Iterative/multi-step = worse (error accumulation)
- **Precision requirements:** Lower bits needed = better (easier to implement)

**For hardware architects:** High amenability patterns are those your hardware should prioritize supporting.

**For neural architects:** High amenability patterns are those you should use for analog deployment.

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd().parent
sys.path.insert(0, str(_ROOT))

import torch
import numpy as np

from neuro_analog.ir.types import AnalogAmenabilityProfile, ArchitectureFamily, DynamicsProfile
from neuro_analog.analysis.compute_amenability import compute_amenability_score
from neuro_analog.analysis.design_heuristics import classify_failure_mode, print_heuristic_report, get_design_recommendations

## AnalogAmenabilityProfile Components

The profile contains all metrics needed for amenability scoring:

In [ ]:
# Create a sample profile for a Neural ODE
neural_ode_profile = AnalogAmenabilityProfile(
    architecture=ArchitectureFamily.NEURAL_ODE,
    model_name="neural_ode_example",
    model_params=50000,
    analog_flop_fraction=0.85,  # 85% of compute in analog
    digital_flop_fraction=0.15,
    hybrid_flop_fraction=0.0,
    da_boundary_count=1,  # Only 1 ADC/DAC crossing
    da_boundary_count_per_step=0,
    layer_count=5,
    sigma_10pct=0.12,  # 12% mismatch threshold from sweep
    min_weight_precision_bits=6,
    min_activation_precision_bits=8,
    dynamics=DynamicsProfile(
        has_dynamics=True,
        dynamics_type="time_varying_ODE",
        num_function_evaluations=10,
    ),
)

print("AnalogAmenabilityProfile components:")
print(f"  Architecture: {neural_ode_profile.architecture.value}")
print(f"  Analog FLOP fraction: {neural_ode_profile.analog_flop_fraction*100:.1f}%")
print(f"  D/A boundaries: {neural_ode_profile.da_boundary_count}")
print(f"  Sigma_10pct: {neural_ode_profile.sigma_10pct*100:.1f}%")
print(f"  Dynamics: {neural_ode_profile.dynamics.dynamics_type}")

## Score Formula

The amenability score is computed as:

```
score = 0.3*analog_frac + 0.3*noise_tolerance + 0.1*precision - 0.2*boundaries - 0.1*dynamics_penalty
```

Each term is normalized to [0, 1].

In [ ]:
# Compute amenability score
score = compute_amenability_score(neural_ode_profile)
print(f"Amenability score: {score:.2f} (0-1, higher = more analog-amenable)")

# Break down the score components
f1 = neural_ode_profile.analog_flop_fraction  # analog FLOP fraction
f2 = min(neural_ode_profile.da_boundary_count / neural_ode_profile.layer_count, 1.0)  # D/A boundaries
f3 = neural_ode_profile.sigma_10pct / 0.15  # sigma_10pct normalized to max tested (15%)
f4 = 0.0  # dynamics penalty (computed internally by dynamics_penalty())
f5 = 1.0 - (neural_ode_profile.min_weight_precision_bits - 2) / (8 - 2)  # precision (2-8 bits to 0-1)
f5 = max(0.0, min(1.0, f5))

print("\nScore breakdown:")
print(f"  f1 (analog FLOP fraction): {f1:.2f} → 0.3*{f1:.2f} = {0.3*f1:.2f}")
print(f"  f2 (D/A boundaries): {f2:.2f} → -0.2*{f2:.2f} = {-0.2*f2:.2f}")
print(f"  f3 (noise tolerance): {f3:.2f} → 0.3*{f3:.2f} = {0.3*f3:.2f}")
print(f"  f4 (dynamics penalty): {f4:.2f} → -0.1*{f4:.2f} = {-0.1*f4:.2f}")
print(f"  f5 (precision): {f5:.2f} → 0.1*{f5:.2f} = {0.1*f5:.2f}")
print(f"  Total: {0.3*f1 + 0.3*f3 + 0.1*f5 - 0.2*f2 - 0.1*f4:.2f}")

## Failure Mode Classification

Architectures are classified into three failure mode buckets:

In [ ]:
# Classify failure mode for Neural ODE
failure_mode = classify_failure_mode(neural_ode_profile)
print(f"Failure mode: {failure_mode}")

# Create profiles for other architectures to see different failure modes
deq_profile = AnalogAmenabilityProfile(
    architecture=ArchitectureFamily.DEQ,
    model_name="deq_example",
    model_params=50000,
    analog_flop_fraction=0.8,
    digital_flop_fraction=0.2,
    da_boundary_count=3,
    layer_count=5,
    sigma_10pct=0.08,  # Lower sigma due to iteration
    dynamics=DynamicsProfile(
        has_dynamics=True,
        dynamics_type="implicit_equilibrium",
    ),
)

diffusion_profile = AnalogAmenabilityProfile(
    architecture=ArchitectureFamily.DIFFUSION,
    model_name="diffusion_example",
    model_params=50000,
    analog_flop_fraction=0.6,
    digital_flop_fraction=0.4,
    da_boundary_count=50,  # Many boundaries across steps
    da_boundary_count_per_step=5,
    layer_count=10,
    sigma_10pct=0.02,  # Very low due to 100+ steps
    dynamics=DynamicsProfile(
        has_dynamics=True,
        dynamics_type="SDE",
        num_diffusion_steps=100,
    ),
)

print("\nFailure modes by architecture:")
print(f"  Neural ODE: {classify_failure_mode(neural_ode_profile)}")
print(f"  DEQ: {classify_failure_mode(deq_profile)}")
print(f"  Diffusion: {classify_failure_mode(diffusion_profile)}")

## Failure Mode Descriptions

In [ ]:
failure_descriptions = {
    "SINGLE_PASS_TOLERANT": "High analog fraction, low boundary density, good noise tolerance. Best for analog.",
    "FIXED_POINT_SENSITIVE": "Iterative convergence compounds mismatch. DEQ architectures fall here.",
    "DIFFUSION_LIKE": "Multi-step pipeline accumulates quantization error. Diffusion models fall here.",
}

print("Failure mode descriptions:")
for mode, desc in failure_descriptions.items():
    print(f"  {mode}: {desc}")

## Heuristic Report

The `print_heuristic_report()` function provides a detailed analysis of the profile:

In [ ]:
# Print heuristic report for Neural ODE
print("Heuristic report for Neural ODE:")
print_heuristic_report(neural_ode_profile)

## Design Recommendations

Based on the failure mode, we can provide design recommendations:

In [ ]:
# Get design recommendations for each failure mode
print("Design recommendations for Neural ODE (single-pass tolerant):")
recommendations = get_design_recommendations(neural_ode_profile)
for rec in recommendations:
    print(f"  - {rec}")

print("\nDesign recommendations for DEQ (fixed-point sensitive):")
recommendations = get_design_recommendations(deq_profile)
for rec in recommendations:
    print(f"  - {rec}")

print("\nDesign recommendations for Diffusion (diffusion-like):")
recommendations = get_design_recommendations(diffusion_profile)
for rec in recommendations:
    print(f"  - {rec}")

## Cross-Architecture Comparison with AnalogTaxonomy

The `AnalogTaxonomy` class enables comparison across multiple architectures:

In [ ]:
from neuro_analog.analysis.taxonomy import AnalogTaxonomy

# Create taxonomy and add profiles
taxonomy = AnalogTaxonomy()
taxonomy.add_profile(neural_ode_profile, has_native_dynamics=True)
taxonomy.add_profile(deq_profile, has_native_dynamics=True)
taxonomy.add_profile(diffusion_profile, has_native_dynamics=True)

# Generate comparison table
table = taxonomy.comparison_table()
print("Cross-architecture comparison:")
print(table)

## Ranking by Analog Amenability

We can rank architectures by their amenability score:

In [ ]:
# Rank by amenability
ranking = taxonomy.rank_by_analog_amenability()
print("Architectures ranked by analog amenability:")
for i, entry in enumerate(ranking, 1):
    print(f"  {i}. {entry.model_name} ({entry.family.value}): {entry.profile.amenability_score:.2f}")

## Radar Chart Visualization

The radar chart shows 6 axes of comparison:

1. **Analog FLOP %** - More = better
2. **D/A Boundary Score** - Fewer boundaries = better
3. **Precision Tolerance** - Lower bits needed = better
4. **Dynamics Naturalness** - ODE/SDE fit to Ark = better
5. **Mismatch Resilience** - Higher sigma threshold = better
6. **Ark Compatibility** - Can generate valid BaseAnalogCkt = better

In [ ]:
from neuro_analog.visualization.comparison_radar import plot_radar_from_taxonomy

# Generate radar chart
fig = plot_radar_from_taxonomy(taxonomy, output_path="amenability_radar.png")
print("Radar chart saved to amenability_radar.png")

## Using Amenability for Co-Design

When choosing an architecture for analog deployment:

1. **Compute amenability score** for candidate architectures
2. **Check failure mode** - single-pass tolerant is best
3. **Review design recommendations** - address specific weaknesses
4. **Compare with taxonomy** - see how it ranks against alternatives
5. **Consider task requirements** - don't sacrifice quality for energy

**For hardware architects:** Use amenability to decide which patterns to support in your hardware. High amenability patterns should be prioritized in your substrate design.

**For neural architects:** Use amenability to decide which patterns to use in your models. High amenability patterns will work better on analog hardware.

**Rule of thumb:** If amenability score > 0.6, the pattern is likely suitable for analog deployment.

## Key Takeaways

1. **Amenability score combines 5 factors:** analog FLOP fraction, noise tolerance, D/A boundaries, dynamics, precision
2. **Score formula:** 0.3*analog_frac + 0.3*noise + 0.1*precision - 0.2*boundaries - 0.1*dynamics
3. **Three failure modes:** single-pass tolerant (best), fixed-point sensitive (DEQ), diffusion-like (worst)
4. **Heuristic reports provide detailed analysis** and design recommendations
5. **AnalogTaxonomy enables cross-arch comparison** and ranking
6. **Radar charts visualize 6 axes** of amenability comparison
7. **Use amenability to guide architecture selection** for energy-efficient deployment
8. **Score > 0.6 indicates good analog suitability**